# Семинар 2. Питон для работы с моделью

**Тема:** базовый питон, без которого не поговорить с моделью — переменные, списки, словари и функции. Учимся читать ответ модели и доставать из него нужное.

На прошлом семинаре мы отправляли запросы и меняли промпты. Сегодня заглянем чуть глубже: **как устроен питон**, на котором всё это работает.

Как и в прошлый раз — **сначала запускаем ячейку, потом смотрим результат, потом меняем одно значение и запускаем снова.**

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sblenlkj/summer-agent-school-seminars-1-2/blob/main/notebooks/seminar_2_8_9.ipynb)


## 1. План

1. **переменные** — как хранить текст и числа;
2. **f-строки** — как собирать запрос из кусочков;
3. **списки** — как хранить много значений и обходить их циклом;
4. **словари** — как хранить данные парами «ключ — значение» (и почему сообщение модели — это словарь);
5. как **прочитать ответ модели** и достать из него нужное;
6. **функции** — как не повторять один и тот же код.

И всё это — на живом примере: мы постоянно будем что-то спрашивать у модели.


## 2. Настройка — просто запусти

Эти две ячейки технические — они подключают модель (как на прошлом семинаре). **Просто запустите их и идём дальше.** Вставьте ключ, когда попросят.


In [ ]:
import uuid
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.filterwarnings("ignore", category=InsecureRequestWarning)

AUTH_URL = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
BASE_URL = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"
MODEL_NAME = "GigaChat-2"

AUTH_KEY = input("Вставьте ключ авторизации GigaChat и нажмите Enter: ")


def get_access_token(auth_key):
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json",
        "RqUID": str(uuid.uuid4()),
        "Authorization": f"Basic {auth_key}",
    }
    response = requests.post(AUTH_URL, headers=headers, data={"scope": "GIGACHAT_API_CORP"}, verify=False)
    print("Статус ответа:", response.status_code)
    if response.status_code != 200:
        print(response.text)
        return None
    return response.json()["access_token"]


ACCESS_TOKEN = get_access_token(AUTH_KEY)
print("Токен получен!" if ACCESS_TOKEN else "Токен не получен, проверьте ключ.")

In [ ]:
# Функция ask_model — отправляет вопрос модели и возвращает текст ответа.
# Сегодня мы поймём, как она устроена внутри. Пока просто запустите.
def ask_model(prompt, temperature=0.7, max_tokens=512):
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)
    if response.status_code != 200:
        print("Ошибка запроса:", response.status_code)
        return None
    return response.json()["choices"][0]["message"]["content"]


print(ask_model("Скажи коротко: всё работает?"))

## 3. Переменные

**Переменная** — это коробочка с именем, в которой что-то лежит. В неё можно положить текст или число, а потом обращаться к ней по имени.

Запустите ячейку и посмотрите, что в коробочках.


In [ ]:
name = "Дима"      # текст (его берут в кавычки) — это строка
age = 14            # число (без кавычек)
school_subject = "информатика"

print(name)
print(age)
print("Любимый предмет:", school_subject)

Главная польза переменной: положили один раз — используем много раз.

Положим в переменную **целый промпт**, а потом отправим его модели.


In [ ]:
my_prompt = "Объясни простыми словами, что такое переменная в программировании."

print("Мы отправим модели такой запрос:")
print(my_prompt)
print()
print("Ответ модели:")
print(ask_model(my_prompt))

## 4. f-строки: собираем запрос из кусочков

Часто нужно собрать текст из нескольких переменных. Для этого в питоне есть **f-строка**: ставим букву `f` перед кавычками, а переменные пишем в `{фигурных скобках}`.

Посмотрите: мы складываем `name` и `topic` прямо внутри текста.


In [ ]:
name = "Аня"
topic = "что такое токен"

prompt = f"Меня зовут {name}. Объясни мне, {topic}, простыми словами и коротко."

print("Получился такой промпт:")
print(prompt)
print()
print(ask_model(prompt))

### Мини-задание 1

Поменяйте значения переменных `name` и `topic` ниже (только текст в кавычках) и запустите. Промпт соберётся сам.


In [ ]:
name = "TODO ваше имя"
topic = "TODO про что спросить, например: что такое промпт"

prompt = f"Меня зовут {name}. Объясни мне, {topic}, простыми словами и коротко."
print(ask_model(prompt))

## 5. Списки

**Список** — это коробка, в которой лежит сразу **много значений по порядку**. Список пишут в квадратных скобках `[ ]`.

Запустите и посмотрите, как доставать элементы и считать их количество.


In [ ]:
terms = ["переменная", "список", "словарь", "функция"]

print("Весь список:", terms)
print("Первый элемент:", terms[0])   # счёт идёт с нуля!
print("Второй элемент:", terms[1])
print("Сколько всего:", len(terms))

Самое удобное в списке — его можно **обойти циклом** `for`: взять каждый элемент по очереди.

Сейчас для **каждого** слова из списка мы попросим у модели короткое объяснение.


In [ ]:
terms = ["переменная", "список", "словарь"]

for term in terms:
    prompt = f"Объясни термин «{term}» одним предложением, для новичка."
    print("=" * 60)
    print("Слово:", term)
    print(ask_model(prompt))

### Мини-задание 2

Добавьте в список `terms` ещё 1-2 слова (в кавычках, через запятую) и запустите. Цикл сам спросит про каждое.


In [ ]:
terms = ["цикл", "функция"]   # TODO: добавьте свои слова

for term in terms:
    prompt = f"Объясни термин «{term}» одним предложением, для новичка."
    print("=" * 60)
    print("Слово:", term)
    print(ask_model(prompt))

## 6. Словари

**Словарь** хранит данные парами **«ключ — значение»**. Пишут в фигурных скобках `{ }`. Это как настоящий словарь: по слову (ключу) находим перевод (значение).

Запустите и посмотрите, как достать значение по ключу.


In [10]:
student = {
    "name": "Дима",
    "age": 14,
    "city": "Москва",
}

print("Весь словарь:", student)
print("Имя:", student["name"])    # достаём значение по ключу
print("Возраст:", student["age"])

Весь словарь: {'name': 'Дима', 'age': 14, 'city': 'Москва'}
Имя: Дима
Возраст: 14


### Cообщение модели — это тоже словарь

Помните, на прошлом семинаре в запросе была вот такая строчка?

```python
{"role": "user", "content": "Привет!"}
```

Это **словарь** с двумя ключами:

- `role` — **роль**, кто говорит (`user` — пользователь, то есть мы; `system` — инструкция модели; `assistant` — ответ модели);
- `content` — сам **текст** сообщения.

То есть «роль» и «инструкция», про которые шла речь на лекции про промпты, — это просто ключи в обычном словаре. Запустите ячейку.


In [11]:
message = {
    "role": "user",
    "content": "Сколько будет 2 + 2?",
}

print("Роль:", message["role"])
print("Текст:", message["content"])

Роль: user
Текст: Сколько будет 2 + 2?


## 7. Список словарей = диалог с моделью

Теперь соединим списки и словари. Если положить **несколько сообщений-словарей в список**, получится `messages` — целый диалог.

- `system` — задаёт, **как** модель себя ведёт;
- `user` — наш вопрос.

Запустите. Функция `ask_chat` отправляет такой список сообщений модели.


In [12]:
def ask_chat(messages, temperature=0.7, max_tokens=512):
    payload = {"model": MODEL_NAME, "messages": messages,
               "temperature": temperature, "max_tokens": max_tokens}
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)
    if response.status_code != 200:
        print("Ошибка запроса:", response.status_code)
        return None
    return response.json()["choices"][0]["message"]["content"]


messages = [
    {"role": "system", "content": "Ты весёлый учитель. Отвечай очень коротко и с примером."},
    {"role": "user", "content": "Что такое список в программировании?"},
]

print(ask_chat(messages))

Список — это набор элементов, записанных в определённом порядке. Например: [яблоко, груша, апельсин].


### Мини-задание 3

Поменяйте **только текст** в сообщении с ролью `system` (например: «Ты очень строгий учитель» или «Ты студент ВШЭ) и запустите. Посмотрите, как изменится стиль ответа — а вопрос остался тот же.


In [13]:
messages = [
    {"role": "system", "content": "TODO опишите, как модель должна себя вести"},
    {"role": "user", "content": "Что такое список в программировании?"},
]

print(ask_chat(messages))

В программировании **список** (иногда называемый массивом или векторной структурой данных) — это структура данных, предназначенная для хранения упорядоченного набора элементов. Основные характеристики списка:

1. **Упорядоченность**: элементы хранятся и извлекаются по определенному порядку.
2. **Доступность по индексу**: каждый элемент имеет уникальный порядковый номер (индекс), начиная с нуля или единицы (в зависимости от языка программирования).
3. **Изменяемость**: обычно списки поддерживают операции вставки, удаления и изменения отдельных элементов.
4. **Типизация**: элементы могут быть разных типов данных (например, целые числа, строки, объекты, другие структуры данных).
5. **Размер гибкость**: размер списка может изменяться динамически: добавлять новые элементы или удалять существующие без необходимости заранее выделять фиксированный объем памяти.

Примеры реализации списков в популярных языках программирования:
- В Python используется тип данных `list`.
- В JavaScript есть встро

## 8. Читаем ответ модели

Теперь самое важное умение семинара — **достать из ответа модели именно то, что нужно**.

Когда модель отвечает, она присылает не просто текст, а **JSON** — это вложенные друг в друга **словари и списки**. То есть всё то, что мы уже знаем!

Сначала посмотрим на «сырой» ответ.


In [14]:
payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Назови один интересный факт о космосе."}],
}
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)
data = response.json()   # превращаем ответ в словари и списки

print("Это словарь. Вот его ключи:")
print(data.keys())

Это словарь. Вот его ключи:
dict_keys(['choices', 'created', 'model', 'object', 'usage'])


Внутри `data` лежит ключ `choices` — это **список** вариантов ответа. Берём первый вариант, в нём словарь `message`, а в нём ключ `content` — наш текст.

Давайте достанем текст **по шагам**, чтобы увидеть, как устроен «адрес».


In [15]:
choices = data["choices"]          # это список вариантов ответа
first = choices[0]                  # берём первый вариант (счёт с нуля)
message = first["message"]          # внутри — словарь сообщения
text = message["content"]           # а в нём — сам текст

print("Текст ответа:")
print(text)
print()
print("Сколько токенов потрачено:", data["usage"])

Текст ответа:
Интересный факт: самая горячая планета Солнечной системы — Венера, хотя Меркурий находится ближе всего к Солнцу. Средняя температура поверхности Венеры достигает около **465 °C**, даже несмотря на то, что она дальше от Солнца, чем раскалённый Меркурий. Такая высокая температура обусловлена плотной атмосферой планеты, состоящей преимущественно из углекислого газа, который создаёт мощный парниковый эффект.

Сколько токенов потрачено: {'prompt_tokens': 20, 'completion_tokens': 84, 'total_tokens': 104, 'precached_prompt_tokens': 2}


Те же самые шаги можно записать **в одну строчку** — это и есть та «длинная» строка из прошлого семинара:

```python
data["choices"][0]["message"]["content"]
```

Теперь понятно, что это просто: зайди в словарь `data` → в список `choices` → возьми первый элемент → в нём словарь `message` → достань ключ `content`.


In [16]:
# Та же самая запись, но в одну строку:
answer = data["choices"][0]["message"]["content"]
print(answer)

Интересный факт: самая горячая планета Солнечной системы — Венера, хотя Меркурий находится ближе всего к Солнцу. Средняя температура поверхности Венеры достигает около **465 °C**, даже несмотря на то, что она дальше от Солнца, чем раскалённый Меркурий. Такая высокая температура обусловлена плотной атмосферой планеты, состоящей преимущественно из углекислого газа, который создаёт мощный парниковый эффект.


## 9. Функции

Мы уже несколько раз писали почти одинаковый код: собрать запрос → отправить → достать текст. Чтобы **не повторяться**, такой код прячут в **функцию**.

Функция — это как своя личная команда. У неё есть:

- имя (например, `my_ask`);
- то, что ей передают (вопрос);
- то, что она возвращает (ответ).

Запустите — ячейка готовит функцию.


In [17]:
def my_ask(question):
    messages = [{"role": "user", "content": question}]
    answer = ask_chat(messages)
    return answer


print("Функция my_ask готова.")

Функция my_ask готова.


Теперь её можно вызывать сколько угодно раз — коротко и удобно. Именно так и устроена `ask_model`, которой мы пользовались.


In [18]:
print(my_ask("Объясни, зачем нужны функции в программировании. Коротко."))
print("-" * 60)
print(my_ask("Приведи смешной пример из жизни про функции."))

Функции нужны, чтобы:

- **Разделять код** — упрощают структуру программы, делая её понятнее и проще для чтения и поддержки.
- **Повторное использование кода** — позволяют повторно применять одни и те же блоки кода в разных частях программы.
- **Улучшают читаемость и поддерживаемость** — каждая функция решает конкретную задачу, облегчая понимание программы.
- **Обеспечивают модульность** — отдельные части программы становятся независимыми модулями, которые легко тестировать и заменять.
------------------------------------------------------------
Вот такой забавный случай с функциями:

---

**Ситуация:**  
Приходит программист Вася домой после работы и рассказывает жене:  
— Сегодня написал новую функцию! Она такая классная — выводит список покупок прямо в холодильник!

Жена удивленно поднимает бровь:  
— А зачем она тебе вообще нужна? У тебя же уже есть функция «список покупок», написанная месяц назад?

Вася смущенно улыбается:  
— Ну, та функция только список печатала, а эта ещё и сам

## 10. Системный промпт

Сейчас наша функция `my_ask()` просто отправляет вопрос модели.

Но иногда нам нужно не только задать вопрос, а ещё объяснить модели, **как именно она должна отвечать**.

Например:

- объясняй как учитель;
- отвечай коротко;
- приводи примеры;
- говори простыми словами;
- веди себя как помощник по Python.

Такая инструкция называется **системный промпт**.

Обычный вопрос пишет пользователь — это роль `user`.

А главная инструкция для модели пишется с ролью `system`.

In [19]:
messages = [
    {
        "role": "system",
        "content": "Ты добрый учитель информатики. Объясняй очень просто, как для ученика 8 класса."
    },
    {
        "role": "user",
        "content": "Что такое функция в программировании?"
    }
]

answer = ask_chat(messages)
print(answer)

Представь себе, что ты хочешь приготовить вкусный пирог. Для этого тебе нужны мука, сахар и яйца — это ингредиенты.

В программировании функции работают точно так же: они помогают собрать программу из отдельных частей или "ингредиентов". 

Функция — это блок кода (инструкция), который выполняет определённую задачу и может быть использован несколько раз в программе. Когда программист пишет код, он разбивает его на маленькие кусочки, чтобы было проще понять и исправить ошибки.

Например:

1. Функция `приветствие` выводит сообщение «Привет!».
2. Функция `сложение` складывает два числа.
3. Функция `вывод_результата` показывает результат вычислений.

Таким образом, программы становятся понятнее и удобнее для работы.


Один и тот же вопрос можно задать с разными системными промптами.

Вопрос не меняется, но меняется роль модели.

In [21]:
question = "Что такое список в Python?"

system_prompt_1 = "Ты учитель информатики. Объясняй просто и спокойно."
system_prompt_2 = "Ты строгий экзаменатор. Отвечай коротко и официально."
system_prompt_3 = "Ты пират. Объясняй весело, но понятно."

for system_prompt in [system_prompt_1, system_prompt_2, system_prompt_3]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    print("=" * 60)
    print("Системный промпт:")
    print(system_prompt)
    print()
    print("Ответ модели:")
    print(ask_chat(messages))

Системный промпт:
Ты учитель информатики. Объясняй просто и спокойно.

Ответ модели:
Список в Python — это особый тип данных, который позволяет хранить в себе набор элементов разного типа (числа, строки, другие списки и т.д.). 

Представь, что ты собираешь коллекцию марок или открыток: каждая марка или открытка — это отдельный элемент списка.
Ящики с марками можно сравнить со списком в Python. Вот как выглядит простой пример:

```python
марки = ["Россия", "Франция", "Германия"]
```

Здесь марки хранятся в списке под названием «марки». Каждый элемент списка записывается внутри квадратных скобок [], а элементы разделяются запятыми.

Основные свойства списков:
- Элементы могут быть разных типов.
- Порядок элементов сохраняется.
- Можно добавлять новые элементы, удалять старые, изменять их местами и многое другое.

Списки очень удобны для работы с данными, потому что позволяют легко манипулировать содержимым и быстро находить нужные элементы.
Системный промпт:
Ты строгий экзаменатор. Отвеч

### Мини-задание 4

Поменяйте системный промпт так, чтобы модель стала вашим помощником.

Например:

- репетитором по математике;
- помощником по английскому;
- учителем Python;
- тренером, который мотивирует учиться;
- персонажем из игры или мультфильма.

Вопрос можно оставить готовый или придумать свой.

In [22]:
system_prompt = "TODO напишите, кем должна быть модель"
question = "TODO напишите вопрос"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

answer = ask_chat(messages)
print(answer)

Какими качествами и навыками должна обладать идеальная модель для успешной карьеры в современной индустрии моды?


## 11. Создаём первого агента

Теперь мы можем сделать своего первого агента.

Агент — это программа, которая:

1. получает вопрос;
2. добавляет инструкцию для модели;
3. отправляет всё модели;
4. возвращает ответ.

Агент = модель + роль + функция.

In [23]:
def first_agent(question):
    system_prompt = "Ты помощник для школьника. Объясняй просто, коротко и с примером."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    answer = ask_chat(messages)
    return answer

Теперь можно вызвать агента как обычную функцию.

Мы передаём вопрос, а агент сам добавляет системный промпт.

In [24]:
print(first_agent("Что такое переменная?"))

Переменная — это "ящичек" в программе, куда можно положить какое-то значение.
равно
например:
x = 5

здесь x — переменная, а число 5 — её значение.


Сделаем агента чуть удобнее.

Пусть он получает не только вопрос, но и роль.

Так мы сможем быстро создавать разных агентов.

In [25]:
def agent(role, question):
    messages = [
        {"role": "system", "content": role},
        {"role": "user", "content": question}
    ]

    answer = ask_chat(messages)
    return answer

Проверим разных агентов.

Функция одна и та же, но роли разные.

In [ ]:
teacher_role = "Ты учитель информатики. Объясняй просто и с примером."
english_role = "Ты помощник по английскому. Переводи слова и объясняй их простыми словами."
idea_role = "Ты генератор идей. Придумывай интересные идеи для школьных проектов."

print(agent(teacher_role, "Что такое цикл for?"))
print("-" * 60)

print(agent(english_role, "Переведи слово variable и объясни его."))
print("-" * 60)

print(agent(idea_role, "Придумай идею проекта про искусственный интеллект."))

### Мини-задание 5

Создайте своего агента.

Нужно заполнить:

- `my_agent_role` — кем будет агент;
- `my_question` — что вы у него спросите.

Примеры:

- агент-репетитор;
- агент-переводчик;
- агент для идей;
- агент, который объясняет мемы;
- агент, который помогает готовиться к контрольной.

In [ ]:
my_agent_role = "TODO опишите, кем будет ваш агент и как он должен отвечать"
my_question = "TODO задайте вопрос вашему агенту"

print(agent(my_agent_role, my_question))